# FFTW C2C F2PY Sequential

In [41]:
#=-----------------------------------------------------------------------=#

In [10]:
%%writefile nc2cs.f90
subroutine fs(ss, tt)
    use, intrinsic :: iso_c_binding
    include "fftw3.f03"
    double complex, intent(out) :: ss
    double precision, intent(out) :: tt
    integer, parameter :: N = 500, M = N, L = N
    type(C_PTR) :: plan, cdata
    complex(C_DOUBLE_COMPLEX), pointer :: data(:,:,:)
    complex(C_DOUBLE_COMPLEX) :: s
    integer :: i, j, k
    double precision :: r, t1, t2
    
    call cpu_time(t1)    ! <--- time measurement
           
    ! in-place transform (note dimension reversal)
    cdata = fftw_alloc_complex(int(L * M * N, C_SIZE_T))
    call c_f_pointer(cdata, data, [L, M, N])

    ! create plan for in-place forward DFT (note dimension reversal)
    
    plan = fftw_plan_dft_3d(N, M, L, data, data, &
                            FFTW_FORWARD, FFTW_ESTIMATE)

    ! initialize data    
     do k = 1, N
        do j = 1, M
          do i = 1, L
            r = ((i-1)*M*L + (j-1)*M  + (k-1) + 1)*1E-6
            data(i, j, k) = cmplx(r, 0, 8)
          enddo
        enddo
     enddo

    ! compute transform (as many times as desired)    
    call fftw_execute_dft(plan, data, data)
    s = sum(data)
    
    call fftw_destroy_plan(plan)
    call fftw_free(cdata)
  
    call cpu_time(t2)    ! <--- time measurement
    
    ! result
    ss = s
    tt = t2-t1
end subroutine

Overwriting nc2cs.f90


In [12]:
%%bash
module load  mathlibs/fftw/3.3.8_openmpi-3.1_gnu
dir=/scratch/app/mathlibs/fftw/3.3.8_openmpi-3.1_gnu
f2py  -c nc2cs.f90  -m nc2cs \
-L$dir/lib  -lfftw3_mpi  -lfftw3  -lm  -I$dir/include \
-DNPY_NO_DEPRECATED_API=NPY_1_7_API_VERSION

running build
running config_cc
unifing config_cc, config, build_clib, build_ext, build commands --compiler options
running config_fc
unifing config_fc, config, build_clib, build_ext, build commands --fcompiler options
running build_src
build_src
building extension "nc2cs" sources
f2py options: []
f2py:> /tmp/tmpfdpi3900/src.linux-x86_64-3.8/nc2csmodule.c
creating /tmp/tmpfdpi3900/src.linux-x86_64-3.8
Reading fortran codes...
	Reading file 'nc2cs.f90' (format:free)
{'before': '', 'this': 'use', 'after': ', intrinsic :: iso_c_binding '}
Line #2 in nc2cs.f90:"    use, intrinsic :: iso_c_binding "
	analyzeline: Could not crack the use statement.
Line #3 in nc2cs.f90:"    include "fftw3.f03" "
	readfortrancode: could not find include file 'fftw3.f03' in . Ignoring.
Post-processing...
	Block: nc2cs
			Block: fs
Post-processing (stage 2)...
Building modules...
	Building module "nc2cs"...
		Constructing wrapper function "fs"...
		  ss,tt = fs()
	Wrote C/API module "nc2cs" to file "/tmp/tmpfdp

In [26]:
%%writefile test01.py
import nc2cs
print(nc2cs.__doc__)

Overwriting test01.py


In [27]:
%%bash
module load  mathlibs/fftw/3.3.8_openmpi-3.1_gnu
python test01.py

This module 'nc2cs' is auto-generated with f2py (version:2).
Functions:
  ss,tt = fs()
.


In [30]:
%%writefile test02.py
import nc2cs
s, t = nc2cs.fs()
print('S: {:.2f}    T: {:.4f} s'.format(s, t))

Overwriting test02.py


In [31]:
%%bash
module load  mathlibs/fftw/3.3.8_openmpi-3.1_gnu
python test02.py

S: 125.00+0.00j    T: 6.8579 s


Copy to /scratch

In [32]:
%%bash
mv test02.py nc2cs_c.py
cp nc2cs* /scratch${PWD#"/prj"}

## Batch script

In [2]:
%%writefile nc2cs.srm
#!/bin/bash
# 1,0 UA partitions:
#   cpu_dev  : 20 min.,  1-4  nodes, 1/1   tasks
#   cpu_small: 72 hours, 1-20 nodes, 16/96 tasks
#SBATCH --partition cpu_small  # Select partition
#SBATCH --ntasks=1             # Total tasks
#SBATCH --job-name nc2cs       # Job name
#SBATCH --time=00:01:00        # Limit execution time
#SBATCH --exclusive            # Exclusive acccess to nodes

echo '========================================'
echo '- Job ID:' $SLURM_JOB_ID
echo '- Tasks per node:' $SLURM_NTASKS_PER_NODE
echo '- # of nodes in the job:' $SLURM_JOB_NUM_NODES
echo '- # of tasks:' $SLURM_NTASKS
echo '- Dir from which sbatch was invoked:' ${SLURM_SUBMIT_DIR##*/}
cd $SLURM_SUBMIT_DIR
echo -n '- List of nodes allocated to the job: '
nodeset -e $SLURM_JOB_NODELIST

# Environment
cd
cd /scratch${PWD#"/prj"}/
module load mathlibs/fftw/3.3.8_openmpi-3.1_gnu
source /scratch/app/anaconda3/2020.11/etc/profile.d/conda.sh
conda activate ./env2
cd fft

# Executable
EXEC="python nc2cs_c.py"

# Start
echo '$ srun -n' $SLURM_NTASKS ${EXEC##*/}
echo '-- output -----------------------------'
srun -n $SLURM_NTASKS $EXEC
echo '~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~'

Overwriting nc2cs.srm


Submit batch script

In [3]:
! sbatch --partition=cpu_dev nc2cs.srm

Submitted batch job 878694


In [4]:
! squeue -n nc2cs -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS
            878694    cpu_dev   R  0:01     1   24


In [6]:
! squeue -n nc2cs -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS


In [7]:
! cat /scratch${PWD#"/prj"}/slurm-878694.out

- Job ID: 878694
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 1
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1285
$ srun -n 1 python nc2cs_c.py
-- output -----------------------------
S: 125.00+0.00j    T: 8.6922 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


In [8]:
! sbatch --partition=cpu_dev nc2cs.srm

Submitted batch job 878718


In [9]:
! squeue -n nc2cs -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS
            878718    cpu_dev   R  0:00     1   24


In [11]:
! squeue -n nc2cs -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS


In [12]:
! cat /scratch${PWD#"/prj"}/slurm-878718.out

- Job ID: 878718
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 1
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1285
$ srun -n 1 python nc2cs_c.py
-- output -----------------------------
S: 125.00+0.00j    T: 8.6367 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


In [13]:
! sbatch --partition=cpu_dev nc2cs.srm

Submitted batch job 878722


In [14]:
! squeue -n nc2cs -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS
            878722    cpu_dev   R  0:04     1   24


In [16]:
! squeue -n nc2cs -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS


In [17]:
! cat /scratch${PWD#"/prj"}/slurm-878722.out

- Job ID: 878722
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 1
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1289
$ srun -n 1 python nc2cs_c.py
-- output -----------------------------
S: 125.00+0.00j    T: 8.6645 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
